In [ ]:
import os
from pathlib import Path

if Path.cwd().name == "research":
    os.chdir("..")

from doctalk.logger import setup_logging
setup_logging()

%load_ext autoreload
%autoreload 2

from dotenv import load_dotenv
load_dotenv()

print("key loaded:", os.environ.get("GOOGLE_API_KEY") is not None)

In [ ]:
from doctalk.config.configuration import ConfigurationManager
from doctalk.components.vector_store import VectorStore
from doctalk.components.retriever import RetrieverBuilder, format_docs

cm = ConfigurationManager()

# LOAD the existing test_session store - not build. no re-embedding.
vector_store = VectorStore(config=cm.get_vector_store_config())
store = vector_store.load(session_id="test_session")

# wrap it as a retriever
retriever = RetrieverBuilder(config=cm.get_retriever_config()).build(store)

print("retriever ready")

In [14]:
question = "How many layers does the encoder have?"

docs = retriever.invoke(question)
print("retrieved", len(docs), "chunks")
print("=" * 60)
print(format_docs(docs))

[2026-07-29 19:38:41,569] INFO - _client - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"
retrieved 4 chunks
[page 3] Figure 1: The Transformer - model architecture.
The Transformer follows this overall architecture using stacked self-attention and point-wise, fully
connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1,
respectively.
3.1 Encoder and Decoder Stacks
Encoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two
sub-layers. The first is a multi-head self-attention mechanism, and the second is a simple, position-
wise fully connected feed-forward network. We employ a residual connection [11] around each of
the two sub-layers, followed by layer normalization [ 1]. That is, the output of each sub-layer is
LayerNorm(x + Sublayer(x)), where Sublayer(x) is the function implemented by the sub-layer
itself. To facilitate thes

In [ ]:
question = "How many layers does the encoder have?"

for k in (4, 8, 12):
    results = store.similarity_search(question, k=k)
    # does any retrieved chunk actually contain the literal answer?
    hit = any("N = 6" in d.page_content or "N= 6" in d.page_content
              or "six identical" in d.page_content.lower() for d in results)
    print(f"k={k:2d}  ->  answer chunk retrieved: {hit}")

In [ ]:
from doctalk.pipelines.stage_01_ingestion import IngestionPipeline
from doctalk.entity import DataIngestionConfig
from doctalk.components.data_ingestion import DataIngestion

# re-chunk the SAME pdf at a smaller size, in memory, without touching config.yaml
di_config = cm.get_data_ingestion_config()
small_config = DataIngestionConfig(
    root_dir=di_config.root_dir,
    chunk_size=500,
    chunk_overlap=100,
)
small_chunks = DataIngestion(config=small_config).run(Path("eval/test_document.pdf"))

# find which chunk holds the answer, and how focused it is
for i, c in enumerate(small_chunks):
    if "N = 6" in c.page_content and "encoder" in c.page_content.lower():
        print(f"answer in chunk {i} (page {c.metadata['page']}), length {len(c.page_content)}")
        print("-" * 40)
        print(c.page_content)
        break

In [ ]:
# how many chunks are actually in the collection?
print("total in collection:", store._collection.count())